# EMG Sensor & Dataset Validation

Works with any recording from the EMG device (`emg_data_*.csv`, optionally with `metadata_*.txt`).

**How to use**
1. `Runtime -> Run all`
2. When asked, upload the `emg_data_*.csv` file (and its `metadata_*.txt` if you have it).
3. At the end, the PDF report and a ZIP (CSV tables + all figures + PDF) are downloaded.

Sampling rate, channels, labels, trials and expected sample count are detected automatically.
To override anything (sampling rate, mains 50/60 Hz, label aliases, quality thresholds, ...) edit the
**USER SETTINGS** block in the *Configuration* cell.

In [ ]:
# ============================================
# EMG SENSOR + DATASET VALIDATION
# Cell 1: Imports
# ============================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import signal
from scipy.stats import variation
from pathlib import Path
import os
import warnings

warnings.filterwarnings("ignore")

print("Libraries loaded successfully.")

In [ ]:
# ============================================
# Figure Auto-Capture (run this right after the imports)
# Every plt.show() below also saves the figure as a PNG so the
# PDF report can include ALL generated images.
# ============================================

import re
import matplotlib.pyplot as plt
from pathlib import Path

OUTPUT_DIR = Path("EMG_VALIDATION_OUTPUT")
FIG_DIR = OUTPUT_DIR / "figures"

saved_figures = []  # list of (title, png_path) in the order generated

_original_show = plt.show


def _show_and_save(*args, **kwargs):
    for num in plt.get_fignums():
        fig = plt.figure(num)

        # skip empty figures and figures already saved
        if not fig.axes or getattr(fig, "_emg_saved", False):
            continue

        title = (
            fig._suptitle.get_text()
            if fig._suptitle is not None
            else fig.axes[0].get_title()
        ) or "Figure"

        slug = re.sub(r"[^A-Za-z0-9]+", "_", title).strip("_")[:60]
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        path = FIG_DIR / f"{len(saved_figures) + 1:03d}_{slug}.png"

        fig.savefig(path, dpi=150, bbox_inches="tight")
        fig._emg_saved = True
        saved_figures.append((title, path))

    return _original_show(*args, **kwargs)


plt.show = _show_and_save

print("Figure auto-capture enabled.")

In [ ]:
# ============================================
# Cell 2: Load an EMG recording (any dataset from the device)
# ============================================
# Upload  emg_data_*.csv  (required)  and  metadata_*.txt  (optional but
# recommended - it provides the sampling rate and expected sample count).
# You can select both files in the same upload dialog.
#
# To use a file on Google Drive / disk instead of uploading, set:
#   CSV_PATH_MANUAL  = "/content/drive/MyDrive/.../emg_data_xxx.csv"
#   META_PATH_MANUAL = "/content/drive/MyDrive/.../metadata_xxx.txt"   (or None)

from google.colab import files

CSV_PATH_MANUAL = None
META_PATH_MANUAL = None

if CSV_PATH_MANUAL:
    CSV_PATH = CSV_PATH_MANUAL
    META_PATH = META_PATH_MANUAL
else:
    uploaded = files.upload()

    csv_files = [f for f in uploaded if f.lower().endswith(".csv")]
    txt_files = [f for f in uploaded if f.lower().endswith(".txt")]

    if not csv_files:
        raise FileNotFoundError("No CSV file uploaded.")

    if len(csv_files) > 1:
        print("More than one CSV uploaded - using the first one:", csv_files[0])

    CSV_PATH = csv_files[0]
    META_PATH = next(
        (f for f in txt_files if "metadata" in f.lower()),
        txt_files[0] if txt_files else None
    )

# If no metadata was given, look for the matching metadata_*.txt next to the CSV
if META_PATH is None:
    guess = Path(CSV_PATH).with_name(
        Path(CSV_PATH).name.replace("emg_data_", "metadata_", 1)
    ).with_suffix(".txt")
    if guess.exists():
        META_PATH = str(guess)

# Parse key=value metadata
META = {}
if META_PATH and Path(META_PATH).exists():
    for line in Path(META_PATH).read_text().splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            META[key.strip()] = value.strip()

df = pd.read_csv(CSV_PATH)

print("Loaded file    :", CSV_PATH)
print("Metadata file  :", META_PATH if META else "none (settings will be auto-detected)")
print("Dataset shape  :", df.shape)

if META:
    print("\nMetadata:")
    for k in ["contributor", "labels", "repeats", "prep_s", "hold_s", "rest_s",
              "sample_rate_hz", "num_samples", "expected_num_samples"]:
        if k in META:
            print(f"  {k:22s} {META[k]}")

display(df.head())

In [ ]:
# ============================================
# Cell 3: Configuration (auto-detected for every dataset)
# ============================================
# Everything below is detected from the CSV / metadata. Edit a USER SETTING
# only if you want to override the automatic choice.

import re

# ---------------- USER SETTINGS ----------------
FS_OVERRIDE = None             # sampling rate in Hz. None -> metadata, else estimated from timestamps
EMG_CHANNEL_COUNT = 8          # first N channels (Ch1..ChN) are EMG; the rest are treated as aux/IMU
ADC_BITS = 12                  # 12-bit ADC -> 0..4095
MAINS_HZ = 50                  # power-line frequency: 50 (BD/EU/Asia) or 60 (US)
REST_LABEL = "rest"            # name of the resting class (case-insensitive)
LABEL_ALIASES = {}             # merge/fix label names, e.g. {"dow": "down", "move_up": "up"}
EXPECTED_SAMPLES_OVERRIDE = None
WINDOW_MS = 200                # ML window length
STRIDE_MS = 50                 # ML window stride

# channel-quality thresholds
MIN_STD = 10                   # below this -> LOW_SIGNAL_VARIATION
MIN_PEAK_TO_PEAK = 50          # below this -> LOW_DYNAMIC_RANGE
MAX_SATURATION_PCT = 0.1       # % of samples at ADC max
MAX_ZERO_PCT = 1               # % of samples at 0
MIN_ACTIVATION_RATIO = 1.2     # gesture RMS / rest RMS
# ------------------------------------------------


def _find_column(candidates):
    lookup = {str(c).strip().lower().replace(" ", "_"): c for c in df.columns}
    for name in candidates:
        if name in lookup:
            return lookup[name]
    return None


# ---- standardise column names so every later cell works on any file ----
rename = {}

for canonical, candidates in {
    "Label": ["label", "gesture", "class"],
    "Trial_ID": ["trial_id", "trial", "repeat"],
    "Timestamp_ms": ["timestamp_ms", "timestamp", "time_ms"],
    "Packet_Number": ["packet_number", "packet", "packet_id"],
}.items():
    found = _find_column(candidates)
    if found is not None and found != canonical:
        rename[found] = canonical

for c in df.columns:
    m = re.fullmatch(r"ch(\d+)", str(c).strip(), flags=re.IGNORECASE)
    if m:
        rename[c] = f"Ch{int(m.group(1))}"

df = df.rename(columns=rename)

LABEL_COLUMN = "Label"
TRIAL_COLUMN = "Trial_ID"
TIMESTAMP_COLUMN = "Timestamp_ms"
PACKET_COLUMN = "Packet_Number"

# ---- channels ----
ALL_CHANNELS = sorted(
    [c for c in df.columns if re.fullmatch(r"Ch\d+", str(c))],
    key=lambda c: int(c[2:])
)

if not ALL_CHANNELS:
    raise ValueError("No channel columns (Ch1, Ch2, ...) found in the CSV.")
if LABEL_COLUMN not in df.columns:
    raise ValueError("No Label column found in the CSV.")

n_emg = min(EMG_CHANNEL_COUNT, len(ALL_CHANNELS))
EMG_CHANNELS = ALL_CHANNELS[:n_emg]
OTHER_CHANNELS = ALL_CHANNELS[n_emg:]

# ---- optional columns ----
if TRIAL_COLUMN not in df.columns:
    print("No trial column found - treating the recording as one trial.")
    df[TRIAL_COLUMN] = 1

HAS_PACKET = PACKET_COLUMN in df.columns
if not HAS_PACKET:
    print("No packet column found - packet integrity checks will be skipped.")
    df[PACKET_COLUMN] = np.nan

# ---- sampling rate: override > metadata > estimated from timestamps ----
FS_ESTIMATED = None
if TIMESTAMP_COLUMN in df.columns:
    _dt = np.diff(df[TIMESTAMP_COLUMN].astype(float).values)
    _dt = _dt[_dt > 0]
    if len(_dt):
        FS_ESTIMATED = 1000 / np.median(_dt)

if FS_OVERRIDE:
    FS_NOMINAL, FS_SOURCE = float(FS_OVERRIDE), "user setting"
elif "sample_rate_hz" in META:
    FS_NOMINAL, FS_SOURCE = float(META["sample_rate_hz"]), "metadata"
elif FS_ESTIMATED:
    FS_NOMINAL, FS_SOURCE = float(round(FS_ESTIMATED)), "estimated from timestamps"
else:
    raise ValueError("Cannot determine the sampling rate. Set FS_OVERRIDE.")

FS_NOMINAL = int(FS_NOMINAL) if float(FS_NOMINAL).is_integer() else FS_NOMINAL

if TIMESTAMP_COLUMN not in df.columns:
    print("No timestamp column found - generating timestamps from the sampling rate.")
    df[TIMESTAMP_COLUMN] = np.arange(len(df)) * 1000 / FS_NOMINAL

if FS_ESTIMATED and abs(FS_ESTIMATED - FS_NOMINAL) / FS_NOMINAL > 0.05:
    print(f"WARNING: timestamps suggest ~{FS_ESTIMATED:.0f} Hz but using "
          f"{FS_NOMINAL} Hz ({FS_SOURCE}).")

# ---- ADC range ----
ADC_MIN = 0
ADC_MAX = 2 ** ADC_BITS - 1

# ---- expected number of samples (only if known) ----
if EXPECTED_SAMPLES_OVERRIDE:
    EXPECTED_SAMPLES = int(EXPECTED_SAMPLES_OVERRIDE)
elif "expected_num_samples" in META:
    EXPECTED_SAMPLES = int(float(META["expected_num_samples"]))
else:
    EXPECTED_SAMPLES = None

# ---- clean labels: strip spaces, apply aliases, merge different capitalisation ----
df[LABEL_COLUMN] = df[LABEL_COLUMN].where(df[LABEL_COLUMN].notna(), "Unlabeled")
df[LABEL_COLUMN] = df[LABEL_COLUMN].astype(str).str.strip()
df[LABEL_COLUMN] = df[LABEL_COLUMN].map(
    lambda s: LABEL_ALIASES.get(s.lower(), s)
)
_spelling = df[LABEL_COLUMN].groupby(df[LABEL_COLUMN].str.lower()).agg(
    lambda s: s.value_counts().index[0]
)
df[LABEL_COLUMN] = df[LABEL_COLUMN].str.lower().map(_spelling)

HAS_REST = (df[LABEL_COLUMN].str.lower() == REST_LABEL.lower()).any()
if not HAS_REST:
    print(f"No '{REST_LABEL}' label found - rest-vs-gesture activation will be NaN.")

# ---- output locations (one folder / report name per dataset) ----
DATASET_NAME = re.sub(r"^emg_data_", "", Path(CSV_PATH).stem)
REPORT_BASENAME = f"EMG_VALIDATION_REPORT_{DATASET_NAME}"
OUTPUT_DIR = Path(f"EMG_VALIDATION_OUTPUT_{DATASET_NAME}")
FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
saved_figures.clear()

# ---- summary ----
print("=" * 70)
print("CONFIGURATION")
print("=" * 70)
print("Dataset        :", DATASET_NAME)
print("Sampling rate  :", FS_NOMINAL, "Hz  (", FS_SOURCE, ")")
print("EMG channels   :", EMG_CHANNELS)
print("Other channels :", OTHER_CHANNELS)
print("ADC range      :", ADC_MIN, "-", ADC_MAX)
print("Mains          :", MAINS_HZ, "Hz")
print("Labels         :", sorted(df[LABEL_COLUMN].unique()))
print("Trials         :", df[TRIAL_COLUMN].nunique())
print("Expected samples:", EXPECTED_SAMPLES if EXPECTED_SAMPLES else "unknown")
print("Output folder  :", OUTPUT_DIR)

In [ ]:
# ============================================
# Cell 4: Basic Dataset Audit
# ============================================

print("=" * 70)
print("DATASET BASIC INFORMATION")
print("=" * 70)

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumns:")
for c in df.columns:
    print(" -", c)

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
missing = df.isna().sum()
display(missing.to_frame("missing_count"))

print("\nTotal missing values:", missing.sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nLabel distribution:")
display(
    df["Label"]
    .value_counts(dropna=False)
    .rename_axis("Label")
    .reset_index(name="Samples")
)

print("\nTrial distribution:")
display(
    df["Trial_ID"]
    .value_counts()
    .sort_index()
    .rename_axis("Trial_ID")
    .reset_index(name="Samples")
)

In [ ]:
# ============================================
# Cell 5: Timestamp / Sampling Analysis
# ============================================

ts = df[TIMESTAMP_COLUMN].astype(float).values

dt_ms = np.diff(ts)

expected_dt_ms = 1000 / FS_NOMINAL

print("=" * 70)
print("TIMESTAMP / SAMPLING ANALYSIS")
print("=" * 70)

print(f"Nominal sampling rate: {FS_NOMINAL} Hz")
print(f"Expected interval: {expected_dt_ms:.3f} ms")

print("\nRaw CSV timestamp statistics:")

print("Minimum dt:", np.min(dt_ms), "ms")
print("Maximum dt:", np.max(dt_ms), "ms")
print("Mean dt:", np.mean(dt_ms), "ms")
print("Median dt:", np.median(dt_ms), "ms")

print("\nNegative timestamp differences:",
      np.sum(dt_ms < 0))

print("Zero timestamp differences:",
      np.sum(dt_ms == 0))

print("Positive timestamp differences:",
      np.sum(dt_ms > 0))

# Positive-only interval analysis
positive_dt = dt_ms[dt_ms > 0]

print("\nPositive interval statistics:")
print("Mean:", np.mean(positive_dt))
print("Median:", np.median(positive_dt))
print("Std:", np.std(positive_dt))

print("\nPercentiles:")
for p in [1, 5, 25, 50, 75, 95, 99]:
    print(f"{p}% = {np.percentile(positive_dt, p):.4f} ms")

In [ ]:
# ============================================
# Cell 6: Timestamp Interval Distribution
# ============================================

plt.figure(figsize=(12, 5))

plt.hist(
    dt_ms,
    bins=200
)

plt.axvline(
    expected_dt_ms,
    linestyle="--",
    linewidth=2,
    label=f"Expected = {expected_dt_ms:.2f} ms"
)

plt.xlabel("Timestamp difference (ms)")
plt.ylabel("Count")
plt.title("EMG Timestamp Interval Distribution")
plt.legend()
plt.grid(True, alpha=0.3)

plt.show()

In [ ]:
# ============================================
# Cell 7: Packet Integrity Analysis
# ============================================

if not HAS_PACKET:
    print("No packet numbers in this file - packet checks skipped.")
    packet_gaps = np.array([])
else:
    packet = df[PACKET_COLUMN].values

    packet_diff = np.diff(packet)

    print("=" * 70)
    print("PACKET ANALYSIS")
    print("=" * 70)

    print("Unique packets:", df[PACKET_COLUMN].nunique())

    print("First packet:", packet[0])
    print("Last packet:", packet[-1])

    print("\nPacket difference counts:")
    display(
        pd.Series(packet_diff)
        .value_counts()
        .sort_index()
        .head(30)
        .to_frame("count")
    )

    # Packet sizes
    packet_sizes = df.groupby(PACKET_COLUMN).size()

    print("\nPacket size statistics:")
    display(packet_sizes.describe().to_frame("value"))

    print("\nMost common packet sizes:")
    display(
        packet_sizes.value_counts()
        .head(20)
        .rename_axis("samples_per_packet")
        .reset_index(name="packet_count")
    )

    # Packet gaps
    packet_gaps = packet_diff[packet_diff > 1]

    print("\nNumber of packet-number gaps:", len(packet_gaps))

    if len(packet_gaps) > 0:
        print("Largest packet-number gap:", packet_gaps.max())
        print("Total missing packet numbers:",
              np.sum(packet_gaps - 1))
    else:
        print("No packet-number gaps detected.")

In [ ]:
# ============================================
# Cell 8: Raw EMG Waveforms
# ============================================

duration_seconds = 20  # length of the plotted section

n_samples_plot = min(
    len(df),
    int(duration_seconds * FS_NOMINAL)
)

time_plot = np.arange(n_samples_plot) / FS_NOMINAL

fig, axes = plt.subplots(
    len(EMG_CHANNELS),
    1,
    figsize=(15, max(3, 2.25 * len(EMG_CHANNELS))),
    sharex=True,
    squeeze=False
)

axes = axes.ravel()

for ax, ch in zip(axes, EMG_CHANNELS):

    ax.plot(
        time_plot,
        df[ch].iloc[:n_samples_plot],
        linewidth=0.7
    )

    ax.set_ylabel(ch)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time (seconds)")

plt.suptitle(
    f"Raw EMG Signals — First {duration_seconds} Seconds",
    fontsize=16
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# Cell 9: EMG Channel Statistics
# ============================================

channel_results = []

for ch in EMG_CHANNELS:

    x = df[ch].astype(float).values

    mean = np.mean(x)
    median = np.median(x)
    std = np.std(x)
    rms = np.sqrt(np.mean(x**2))
    mav = np.mean(np.abs(x - mean))

    minimum = np.min(x)
    maximum = np.max(x)

    peak_to_peak = maximum - minimum

    zero_pct = np.mean(x == ADC_MIN) * 100
    max_pct = np.mean(x == ADC_MAX) * 100

    unique_values = len(np.unique(x))

    channel_results.append({
        "Channel": ch,
        "Mean": mean,
        "Median": median,
        "Std": std,
        "RMS_raw": rms,
        "MAV_centered": mav,
        "Min": minimum,
        "Max": maximum,
        "Peak_to_peak": peak_to_peak,
        "Unique_values": unique_values,
        "Zero_%": zero_pct,
        "Max_ADC_%": max_pct
    })

channel_summary = pd.DataFrame(channel_results)

display(
    channel_summary.round(4)
)

In [ ]:
# ============================================
# Cell 10: Suspicious Channel Detection
# ============================================

print("=" * 70)
print("CHANNEL QUALITY SCREENING")
print("=" * 70)

for _, row in channel_summary.iterrows():

    ch = row["Channel"]

    warnings_list = []

    if row["Std"] < MIN_STD:
        warnings_list.append("VERY_LOW_VARIATION")

    if row["Peak_to_peak"] < MIN_PEAK_TO_PEAK:
        warnings_list.append("LOW_DYNAMIC_RANGE")

    if row["Zero_%"] > MAX_ZERO_PCT:
        warnings_list.append("ZERO_VALUE_PRESENT")

    if row["Max_ADC_%"] > MAX_SATURATION_PCT:
        warnings_list.append("POSSIBLE_SATURATION")

    if row["Unique_values"] < 100:
        warnings_list.append("LOW_NUMBER_OF_UNIQUE_VALUES")

    if warnings_list:
        status = "WARNING"
    else:
        status = "PASS"

    print(
        f"{ch:5s} | {status:7s} | "
        + ", ".join(warnings_list)
    )

In [ ]:
# ============================================
# Cell 11: Rest vs Gesture Activation
# ============================================

is_rest = df["Label"].str.lower() == REST_LABEL.lower()

rest_df = df[is_rest]
gesture_df = df[~is_rest]

activation_results = []


def centered_rms(x):
    if len(x) == 0:
        return np.nan
    x = x - np.mean(x)
    return np.sqrt(np.mean(x ** 2))


for ch in EMG_CHANNELS:

    rest_rms = centered_rms(rest_df[ch].values.astype(float))
    gesture_rms = centered_rms(gesture_df[ch].values.astype(float))

    ratio = (
        gesture_rms / rest_rms
        if np.isfinite(rest_rms) and rest_rms > 0
        else np.nan
    )

    activation_results.append({
        "Channel": ch,
        "Rest_RMS": rest_rms,
        "Gesture_RMS": gesture_rms,
        "Activation_Ratio": ratio
    })

activation_summary = pd.DataFrame(
    activation_results
)

display(
    activation_summary.round(4)
)

In [ ]:
# ============================================
# Cell 12: RMS by Label and Channel
# ============================================

rms_by_label = []

for label in sorted(df["Label"].dropna().unique()):

    subset = df[df["Label"] == label]

    row = {"Label": label}

    for ch in EMG_CHANNELS:

        x = subset[ch].values.astype(float)

        x_centered = x - np.mean(x)

        rms = np.sqrt(
            np.mean(x_centered ** 2)
        )

        row[ch] = rms

    rms_by_label.append(row)

rms_table = pd.DataFrame(rms_by_label)

display(rms_table.round(3))

In [ ]:
# ============================================
# Cell 13: RMS by Gesture
# ============================================

plot_data = rms_table.set_index("Label")

ax = plot_data.plot(
    kind="bar",
    figsize=(15, 7)
)

ax.set_title("EMG RMS by Gesture")
ax.set_xlabel("Gesture")
ax.set_ylabel("Centered RMS")
plt.xticks(rotation=0)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# Cell 14: FFT / PSD Analysis
# ============================================

def plot_psd(channel, label=None, max_freq=None):

    if max_freq is None:
        max_freq = FS_NOMINAL / 2

    if label is None:
        x = df[channel].values.astype(float)
        title = f"{channel} — Entire Recording"
    else:
        x = df[df["Label"] == label][channel].values.astype(float)
        title = f"{channel} — {label}"

    # Remove DC
    x = x - np.mean(x)

    frequencies, psd = signal.welch(
        x,
        fs=FS_NOMINAL,
        nperseg=min(2048, len(x)),
        noverlap=None
    )

    mask = frequencies <= max_freq

    plt.figure(figsize=(12, 5))

    plt.semilogy(
        frequencies[mask],
        psd[mask]
    )

    if MAINS_HZ < max_freq:
        plt.axvline(
            MAINS_HZ,
            linestyle="--",
            linewidth=2,
            label=f"{MAINS_HZ} Hz mains"
        )

    plt.xlabel("Frequency (Hz)")
    plt.ylabel("PSD")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.show()


# PSD of every channel (EMG + aux). To look at one gesture only:
#   plot_psd("Ch1", label="fist_close")
for ch in ALL_CHANNELS:
    plot_psd(ch)

In [ ]:
# ============================================
# Cell 15: Power-Line (Mains) Interference Analysis
# ============================================

def band_power(freqs, psd, low, high):

    mask = (
        (freqs >= low) &
        (freqs <= high)
    )

    if np.sum(mask) < 2:
        return np.nan

    return np.trapezoid(
        psd[mask],
        freqs[mask]
    )


power_results = []

for ch in EMG_CHANNELS:

    x = df[ch].values.astype(float)

    x = x - np.mean(x)

    freqs, psd = signal.welch(
        x,
        fs=FS_NOMINAL,
        nperseg=min(4096, len(x))
    )

    total_power = band_power(
        freqs,
        psd,
        1,
        FS_NOMINAL / 2
    )

    power_50 = band_power(
        freqs,
        psd,
        MAINS_HZ - 2,
        MAINS_HZ + 2
    )

    power_results.append({
        "Channel": ch,
        "Total_Power": total_power,
        f"Power_{MAINS_HZ - 2}_{MAINS_HZ + 2}Hz": power_50,
        f"Mains_{MAINS_HZ}Hz_Power_%": (
            100 * power_50 / total_power
            if total_power > 0
            else np.nan
        )
    })

power_summary = pd.DataFrame(power_results)

display(
    power_summary.round(4)
)

In [ ]:
# ============================================
# Cell 16: Frequency Band Power
# ============================================

bands = {
    "1-10 Hz": (1, 10),
    "10-20 Hz": (10, 20),
    "20-50 Hz": (20, 50),
    "50-100 Hz": (50, 100),
    "100-150 Hz": (100, 150),
    "150-250 Hz": (150, 250)
}

# keep only the bands that fit below the Nyquist frequency
NYQUIST = FS_NOMINAL / 2
bands = {
    name: (low, min(high, NYQUIST))
    for name, (low, high) in bands.items()
    if low < NYQUIST
}

band_results = []

for ch in EMG_CHANNELS:

    x = df[ch].values.astype(float)

    x = x - np.mean(x)

    freqs, psd = signal.welch(
        x,
        fs=FS_NOMINAL,
        nperseg=min(4096, len(x))
    )

    row = {"Channel": ch}

    for band_name, (low, high) in bands.items():

        row[band_name] = band_power(
            freqs,
            psd,
            low,
            high
        )

    band_results.append(row)

band_summary = pd.DataFrame(band_results)

display(band_summary.round(5))

In [ ]:
# ============================================
# Cell 17: ADC Saturation / Clipping
# ============================================

saturation_results = []

for ch in EMG_CHANNELS:

    x = df[ch].values.astype(float)

    zero_count = np.sum(x <= ADC_MIN)
    max_count = np.sum(x >= ADC_MAX)

    saturation_results.append({
        "Channel": ch,
        "Zero_Count": zero_count,
        "Zero_%": 100 * zero_count / len(x),
        "Max_Count": max_count,
        "Max_ADC_%": 100 * max_count / len(x)
    })

saturation_summary = pd.DataFrame(
    saturation_results
)

display(
    saturation_summary.round(6)
)

In [ ]:
# ============================================
# Cell 18: Flatline Detection
# ============================================

flatline_results = []

WINDOW = int(FS_NOMINAL * 1.0)  # 1 second

for ch in EMG_CHANNELS:

    x = df[ch].values.astype(float)

    window_stds = []

    for start in range(
        0,
        len(x) - WINDOW + 1,
        WINDOW
    ):

        segment = x[start:start + WINDOW]

        window_stds.append(
            np.std(segment)
        )

    window_stds = np.array(window_stds)

    flat_windows = np.sum(
        window_stds < 1e-6
    )

    flatline_results.append({
        "Channel": ch,
        "Total_1s_Windows": len(window_stds),
        "Flat_Windows": flat_windows,
        "Flat_Window_%": (
            100 * flat_windows / len(window_stds)
        )
    })

flatline_summary = pd.DataFrame(
    flatline_results
)

display(
    flatline_summary.round(4)
)

In [ ]:
# ============================================
# Cell 19: EMG Channel Correlation
# ============================================

corr = df[EMG_CHANNELS].corr()

plt.figure(figsize=(10, 8))

plt.imshow(
    corr,
    interpolation="nearest",
    aspect="auto"
)

plt.colorbar(label="Correlation")

plt.xticks(
    range(len(EMG_CHANNELS)),
    EMG_CHANNELS
)

plt.yticks(
    range(len(EMG_CHANNELS)),
    EMG_CHANNELS
)

plt.title("EMG Channel Correlation Matrix")

plt.tight_layout()
plt.show()

display(corr.round(3))

In [ ]:
# ============================================
# Cell 20: Trial-Level Analysis
# ============================================

trial_results = []

for trial in sorted(df[TRIAL_COLUMN].unique()):

    trial_df = df[
        df[TRIAL_COLUMN] == trial
    ]

    for label in sorted(
        trial_df["Label"].dropna().unique()
    ):

        subset = trial_df[
            trial_df["Label"] == label
        ]

        row = {
            "Trial_ID": trial,
            "Label": label,
            "Samples": len(subset)
        }

        for ch in EMG_CHANNELS:

            x = subset[ch].values.astype(float)

            x = x - np.mean(x)

            row[f"{ch}_RMS"] = np.sqrt(
                np.mean(x ** 2)
            )

        trial_results.append(row)

trial_summary = pd.DataFrame(trial_results)

display(trial_summary.head(20))

In [ ]:
# ============================================
# Cell 21: Repeatability Analysis
# ============================================

repeatability_results = []

for label in sorted(
    trial_summary["Label"].unique()
):

    label_data = trial_summary[
        trial_summary["Label"] == label
    ]

    for ch in EMG_CHANNELS:

        values = label_data[
            f"{ch}_RMS"
        ].values

        mean_value = np.mean(values)
        std_value = np.std(values, ddof=1)

        if mean_value != 0:
            cv = (
                100 *
                std_value /
                abs(mean_value)
            )
        else:
            cv = np.nan

        repeatability_results.append({
            "Label": label,
            "Channel": ch,
            "Mean_RMS": mean_value,
            "Std_RMS": std_value,
            "CV_%": cv
        })

repeatability = pd.DataFrame(
    repeatability_results
)

display(
    repeatability.round(3)
)

In [ ]:
# ============================================
# Cell 22: Repeatability Plot
# ============================================

for ch in EMG_CHANNELS:

    plt.figure(figsize=(14, 6))

    for label in sorted(
        trial_summary["Label"].unique()
    ):

        subset = trial_summary[
            trial_summary["Label"] == label
        ]

        plt.plot(
            subset["Trial_ID"],
            subset[f"{ch}_RMS"],
            marker="o",
            label=label
        )

    plt.title(
        f"{ch} — RMS Across {df[TRIAL_COLUMN].nunique()} Trial(s)"
    )

    plt.xlabel("Trial")
    plt.ylabel("RMS")
    plt.xticks(
        sorted(df[TRIAL_COLUMN].unique())
    )

    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.show()

In [ ]:
# ============================================
# Cell 23: Label Balance
# ============================================

label_counts = (
    df["Label"]
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(10, 5))

plt.bar(
    label_counts.index,
    label_counts.values
)

plt.xlabel("Label")
plt.ylabel("Number of samples")
plt.title("Dataset Label Distribution")

plt.xticks(rotation=30)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()
plt.show()

display(
    label_counts.to_frame("Samples")
)

In [ ]:
# ============================================
# Cell 24: Label Transition Analysis
# ============================================

label_change = df["Label"].ne(
    df["Label"].shift()
)

transition_indices = np.where(
    label_change.values
)[0]

print("Number of label segments:",
      len(transition_indices))

transition_table = []

for idx in transition_indices:

    if idx == 0:
        previous_label = "START"
    else:
        previous_label = df["Label"].iloc[idx - 1]

    current_label = df["Label"].iloc[idx]

    transition_table.append({
        "Row": idx,
        "Previous": previous_label,
        "Current": current_label,
        "Timestamp_ms": df[TIMESTAMP_COLUMN].iloc[idx],
        "Trial_ID": df[TRIAL_COLUMN].iloc[idx]
    })

transition_df = pd.DataFrame(
    transition_table
)

display(transition_df)

In [ ]:
# ============================================
# Cell 25: Trial × Label Sample Counts
# ============================================

trial_label_counts = pd.crosstab(
    df[TRIAL_COLUMN],
    df["Label"]
)

display(trial_label_counts)

plt.figure(figsize=(12, 6))

trial_label_counts.plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title(
    "Samples per Label for Each Trial"
)

plt.xlabel("Trial")
plt.ylabel("Samples")

plt.xticks(rotation=0)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# Cell 26: ML Windowing Analysis
# ============================================

WINDOW_SAMPLES = int(
    FS_NOMINAL * WINDOW_MS / 1000
)

STRIDE_SAMPLES = int(
    FS_NOMINAL * STRIDE_MS / 1000
)

print("Sampling rate:", FS_NOMINAL)
print("Window:", WINDOW_MS, "ms")
print("Window samples:", WINDOW_SAMPLES)
print("Stride:", STRIDE_MS, "ms")
print("Stride samples:", STRIDE_SAMPLES)

In [ ]:
# ============================================
# Cell 27: Window Label Purity
# ============================================

def calculate_window_purity(
    labels,
    window_size,
    stride
):

    results = []

    for start in range(
        0,
        len(labels) - window_size + 1,
        stride
    ):

        window_labels = labels[
            start:start + window_size
        ]

        counts = pd.Series(
            window_labels
        ).value_counts()

        majority_label = counts.index[0]

        purity = (
            counts.iloc[0] /
            window_size
        )

        results.append({
            "Start": start,
            "Majority_Label": majority_label,
            "Purity": purity,
            "Unique_Labels": len(counts)
        })

    return pd.DataFrame(results)


window_results = calculate_window_purity(
    df["Label"].values,
    WINDOW_SAMPLES,
    STRIDE_SAMPLES
)

print(
    "Total windows:",
    len(window_results)
)

print(
    "Windows with 100% label purity:",
    np.sum(
        window_results["Purity"] == 1.0
    )
)

print(
    "Windows with mixed labels:",
    np.sum(
        window_results["Purity"] < 1.0
    )
)

print(
    "Average window purity:",
    window_results["Purity"].mean()
)

display(
    window_results.head(20)
)

In [ ]:
# ============================================
# Cell 28: Automated Quality Scorecard
# ============================================

quality_rows = []

for ch in EMG_CHANNELS:

    row = channel_summary[
        channel_summary["Channel"] == ch
    ].iloc[0]

    activation_row = activation_summary[
        activation_summary["Channel"] == ch
    ].iloc[0]

    saturation_row = saturation_summary[
        saturation_summary["Channel"] == ch
    ].iloc[0]

    warnings_list = []

    # Low variation
    if row["Std"] < MIN_STD:
        warnings_list.append(
            "LOW_SIGNAL_VARIATION"
        )

    # Very small range
    if row["Peak_to_peak"] < MIN_PEAK_TO_PEAK:
        warnings_list.append(
            "LOW_DYNAMIC_RANGE"
        )

    # Saturation
    if saturation_row["Max_ADC_%"] > MAX_SATURATION_PCT:
        warnings_list.append(
            "ADC_SATURATION"
        )

    # Zero values
    if saturation_row["Zero_%"] > MAX_ZERO_PCT:
        warnings_list.append(
            "ZERO_VALUES"
        )

    # Activation
    activation_ratio = activation_row[
        "Activation_Ratio"
    ]

    if np.isfinite(activation_ratio):
        if activation_ratio < MIN_ACTIVATION_RATIO:
            warnings_list.append(
                "WEAK_REST_GESTURE_SEPARATION"
            )

    status = (
        "WARNING"
        if warnings_list
        else "PASS"
    )

    quality_rows.append({
        "Channel": ch,
        "Status": status,
        "Std": row["Std"],
        "Peak_to_peak": row["Peak_to_peak"],
        "Activation_Ratio": activation_ratio,
        "Zero_%": saturation_row["Zero_%"],
        "Saturation_%": saturation_row["Max_ADC_%"],
        "Warnings": "; ".join(warnings_list)
    })

quality_report = pd.DataFrame(
    quality_rows
)

display(
    quality_report.round(4)
)

In [ ]:
# ============================================
# Cell 29: Acquisition Quality Report
# ============================================

total_samples = len(df)

negative_dt_count = np.sum(dt_ms < 0)
zero_dt_count = np.sum(dt_ms == 0)

expected_samples = int(
    (ts[-1] - ts[0]) /
    (1000 / FS_NOMINAL)
) + 1

acquisition_report = pd.DataFrame([{

    "Rows": len(df),

    "Channels": len(EMG_CHANNELS),

    "Nominal_Fs_Hz": FS_NOMINAL,

    "Expected_dt_ms": expected_dt_ms,

    "Median_positive_dt_ms":
        np.median(positive_dt),

    "Negative_timestamp_diffs":
        negative_dt_count,

    "Duplicate_timestamps":
        df[TIMESTAMP_COLUMN].duplicated().sum(),

    "Duplicate_rows":
        df.duplicated().sum(),

    "Unique_packets":
        df[PACKET_COLUMN].nunique(),

    "Packet_number_gaps":
        len(packet_gaps),

    "Missing_packet_numbers":
        np.sum(packet_gaps - 1)
        if len(packet_gaps) else 0,

    "Start_timestamp_ms":
        ts[0],

    "End_timestamp_ms":
        ts[-1],

    "Duration_seconds":
        (ts[-1] - ts[0]) / 1000,

    "Expected_samples_metadata":
        EXPECTED_SAMPLES if EXPECTED_SAMPLES else np.nan,

    "Actual_samples":
        len(df),

    "Coverage_vs_metadata_%":
        100 * len(df) / EXPECTED_SAMPLES
        if EXPECTED_SAMPLES else np.nan,

    "Labels":
        ", ".join(
            sorted(df["Label"].unique())
        ),

    "Trials":
        df[TRIAL_COLUMN].nunique()

}])

display(acquisition_report.T)

In [ ]:
# ============================================
# Cell 30: Export Reports
# ============================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

channel_summary.to_csv(
    OUTPUT_DIR / "EMG_CHANNEL_SUMMARY.csv",
    index=False
)

activation_summary.to_csv(
    OUTPUT_DIR / "EMG_ACTIVATION_SUMMARY.csv",
    index=False
)

power_summary.to_csv(
    OUTPUT_DIR / "EMG_FREQUENCY_MAINS_SUMMARY.csv",
    index=False
)

band_summary.to_csv(
    OUTPUT_DIR / "EMG_BAND_POWER.csv",
    index=False
)

saturation_summary.to_csv(
    OUTPUT_DIR / "EMG_SATURATION.csv",
    index=False
)

flatline_summary.to_csv(
    OUTPUT_DIR / "EMG_FLATLINE.csv",
    index=False
)

repeatability.to_csv(
    OUTPUT_DIR / "EMG_REPEATABILITY.csv",
    index=False
)

trial_summary.to_csv(
    OUTPUT_DIR / "EMG_TRIAL_SUMMARY.csv",
    index=False
)

window_results.to_csv(
    OUTPUT_DIR / "EMG_WINDOW_ANALYSIS.csv",
    index=False
)

quality_report.to_csv(
    OUTPUT_DIR / "EMG_CHANNEL_QUALITY_REPORT.csv",
    index=False
)

acquisition_report.to_csv(
    OUTPUT_DIR / "EMG_ACQUISITION_REPORT.csv",
    index=False
)

transition_df.to_csv(
    OUTPUT_DIR / "EMG_LABEL_TRANSITIONS.csv",
    index=False
)

print("Reports saved to:")
print(OUTPUT_DIR)

In [ ]:
# ============================================
# Cell 31: Build the PDF Report
# (summary + all tables + every generated figure)
# ============================================

import textwrap
import datetime
from matplotlib.backends.backend_pdf import PdfPages

PDF_PATH = OUTPUT_DIR / f"{REPORT_BASENAME}.pdf"

PORTRAIT = (8.27, 11.69)
LANDSCAPE = (11.69, 8.27)


def _fmt(v):
    if isinstance(v, (float, np.floating)):
        if not np.isfinite(v):
            return str(v)
        return f"{v:,.4f}".rstrip("0").rstrip(".") if abs(v) < 1e6 else f"{v:,.0f}"
    if isinstance(v, str) and len(v) > 28:
        return textwrap.fill(v, 28)
    return str(v)


def add_text_page(pdf, title, lines):
    fig = plt.figure(figsize=PORTRAIT)
    fig.text(0.07, 0.95, title, fontsize=20, weight="bold", va="top")
    fig.text(
        0.07, 0.88, "\n".join(lines),
        fontsize=11, va="top", family="monospace"
    )
    pdf.savefig(fig)
    plt.close(fig)


def add_table_pages(pdf, title, table, rows_per_page=26, cols_per_page=9):
    """Render a DataFrame as one or more PDF pages (splits long/wide tables)."""
    table = table.copy()
    if table.index.name or not isinstance(table.index, pd.RangeIndex):
        table = table.reset_index()

    if table.empty:
        return

    first_col = table.columns[0]
    other_cols = list(table.columns[1:])
    col_chunks = [
        other_cols[i:i + cols_per_page - 1]
        for i in range(0, max(len(other_cols), 1), cols_per_page - 1)
    ] or [[]]

    n_row_pages = int(np.ceil(len(table) / rows_per_page))

    for c_i, cols in enumerate(col_chunks):
        sub_cols = [first_col] + cols

        for r_i in range(n_row_pages):
            chunk = table.iloc[r_i * rows_per_page:(r_i + 1) * rows_per_page]
            cells = [[_fmt(v) for v in row] for row in chunk[sub_cols].values]

            fig, ax = plt.subplots(figsize=LANDSCAPE)
            ax.axis("off")

            suffix = []
            if n_row_pages > 1:
                suffix.append(f"rows {r_i * rows_per_page + 1}-"
                              f"{r_i * rows_per_page + len(chunk)} of {len(table)}")
            if len(col_chunks) > 1:
                suffix.append(f"columns part {c_i + 1}/{len(col_chunks)}")

            ax.set_title(
                title + (f"  ({', '.join(suffix)})" if suffix else ""),
                fontsize=14, weight="bold", loc="left"
            )

            tbl = ax.table(
                cellText=cells,
                colLabels=[str(c) for c in sub_cols],
                loc="upper center",
                cellLoc="center"
            )
            tbl.auto_set_font_size(False)
            tbl.set_fontsize(8)
            tbl.auto_set_column_width(list(range(len(sub_cols))))
            tbl.scale(1, 1.6)

            for (row, _), cell in tbl.get_celld().items():
                if row == 0:
                    cell.set_facecolor("#dbe4f0")
                    cell.set_text_props(weight="bold")

            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)


def add_image_page(pdf, title, path, number, total):
    img = plt.imread(path)
    h, w = img.shape[:2]
    figsize = LANDSCAPE if w > h else PORTRAIT

    fig = plt.figure(figsize=figsize)
    ax = fig.add_axes([0.04, 0.05, 0.92, 0.86])
    ax.imshow(img)
    ax.axis("off")

    fig.text(0.04, 0.965, f"Figure {number}/{total}: {title}",
             fontsize=12, weight="bold", va="center")
    fig.text(0.96, 0.02, Path(path).name, fontsize=7, ha="right", color="gray")

    pdf.savefig(fig)
    plt.close(fig)


# ---------- tables (only those that exist in this session) ----------

table_specs = [
    ("Acquisition Quality Report", "acquisition_report", lambda t: t.T.reset_index().rename(columns={"index": "Metric", 0: "Value"})),
    ("Automated Channel Quality Scorecard", "quality_report", None),
    ("EMG Channel Statistics", "channel_summary", None),
    ("Rest vs Gesture Activation", "activation_summary", None),
    ("RMS by Gesture and Channel", "rms_table", None),
    ("50 Hz Interference", "power_summary", None),
    ("Frequency Band Power", "band_summary", None),
    ("ADC Saturation / Clipping", "saturation_summary", None),
    ("Flatline Detection", "flatline_summary", None),
    ("Repeatability Across Trials", "repeatability", None),
    ("Trial-Level RMS Summary", "trial_summary", None),
    ("Trial x Label Sample Counts", "trial_label_counts", None),
    ("Channel Correlation Matrix", "corr", None),
    ("Label Transitions", "transition_df", None),
]

# small derived tables
if "label_counts" in globals():
    table_specs.append(
        ("Label Balance", "label_counts", lambda s: s.rename_axis("Label").reset_index(name="Samples"))
    )

if "window_results" in globals():
    window_overview = pd.DataFrame([{
        "Total_windows": len(window_results),
        "Pure_windows_(100%)": int((window_results["Purity"] == 1.0).sum()),
        "Mixed_windows": int((window_results["Purity"] < 1.0).sum()),
        "Average_purity": window_results["Purity"].mean(),
    }])
    table_specs.append(("ML Window Label Purity (overview)", "window_overview", None))

# ---------- assemble the PDF ----------

figures_to_include = list(saved_figures)

with PdfPages(PDF_PATH) as pdf:

    # cover page
    cover = [
        f"Source file : {CSV_PATH}",
        f"Generated   : {datetime.datetime.now():%Y-%m-%d %H:%M}",
        f"Rows        : {len(df):,}",
        f"Duration    : {(ts[-1] - ts[0]) / 1000:.1f} s",
        f"Nominal Fs  : {FS_NOMINAL} Hz ({FS_SOURCE})",
        f"EMG channels: {', '.join(EMG_CHANNELS)}",
        f"Labels      : {', '.join(sorted(df['Label'].dropna().unique()))}",
        f"Trials      : {df[TRIAL_COLUMN].nunique()}",
        f"Figures     : {len(figures_to_include)}",
    ]

    if META.get("contributor"):
        cover.insert(1, f"Contributor : {META['contributor']}")
    if EXPECTED_SAMPLES:
        cover.append(
            f"Samples     : {len(df):,} of {EXPECTED_SAMPLES:,} expected "
            f"({100 * len(df) / EXPECTED_SAMPLES:.1f}%)"
        )

    if "quality_report" in globals():
        n_pass = int((quality_report["Status"] == "PASS").sum())
        n_warn = int((quality_report["Status"] == "WARNING").sum())
        cover += ["", f"Channel quality: {n_pass} PASS, {n_warn} WARNING"]
        for _, r in quality_report[quality_report["Status"] == "WARNING"].iterrows():
            cover.append(f"  {r['Channel']}: {r['Warnings']}")

    add_text_page(pdf, "EMG Sensor & Dataset Validation Report", cover)

    # tables
    for title, var, transform in table_specs:
        if var not in globals():
            continue
        obj = globals()[var]
        table = transform(obj) if transform else obj
        if isinstance(table, pd.Series):
            table = table.to_frame()
        add_table_pages(pdf, title, table)

    # every figure generated in the notebook, in order
    for i, (title, path) in enumerate(figures_to_include, 1):
        add_image_page(pdf, title, path, i, len(figures_to_include))

    info = pdf.infodict()
    info["Title"] = "EMG Sensor & Dataset Validation Report"
    info["CreationDate"] = datetime.datetime.now()

print("PDF report created:", PDF_PATH)
print("Figures included  :", len(figures_to_include))
print("Tables included   :", sum(1 for _, v, _ in table_specs if v in globals()))

In [ ]:
# ============================================
# Cell 32: ZIP Everything (CSV reports + figures + PDF)
# ============================================

import shutil

ZIP_PATH = shutil.make_archive(
    REPORT_BASENAME,
    "zip",
    OUTPUT_DIR
)

print("ZIP created:")
print(ZIP_PATH)

In [ ]:
from google.colab import files

files.download(str(ZIP_PATH))
files.download(str(PDF_PATH))